In [24]:
import simoolator as moo

# Create data to initalze a Cow and run laccurve5_dijkstra

# Create iStateVars
# iBW=650 
#     # Initial bodyweight
# ifBWfat=0.18
#     # ifBWfat=105 - 140 kg (dropped to no lower than 65 kg) Phillips et al., 2003. JDS 86:3534
# iER = iBW * ifBWfat * 9.0
# milkcumul = 0 
# DMIcumul = 0
# IOFCcumul = 0

# iStateVars = [iBW, iER, milkcumul, DMIcumul, IOFCcumul] # must be in same order as in dynamic()
iStateVars = [650, 1053.0, 0, 0, 0]

# Create parameters
parameters_laccurve5 = {
    'a': 18.3085866,
    'b': 0.05050184,
    'b0': 0.05328367,
    'c': 0.00276552,
    'E': 0.5,
    'L': 0.567,
    'KDMIMILK': 0.372,
    'KDMImbw': 0.0968,
    'Hcfeed': 1.6,
    'Hcmilk': 0.749,
    'KL': 20,
    'expL': 20,
    # 'pregdate': 100,
    'pregdate': 150,
    'kf1': 0,
    'kf2': 0,
    'milk_value': 0.9
}

# Model returns 
output = ['t', 'dijkstra_milk', 'NEmilk', 'milk', 'DMI', 'E', 'BW', 'BFat', 'ER', 'IOFC', 'milkcumul', 'DMIcumul', 'IOFCcumul', 'NEgest']


In [22]:
# Define model functions

def laccurve5_dijkstra(self, stateVars, t, scenario_param=None, prev_var_return=None):
    # prev_var_return is required if you want to have a variable with an initial value that then gets updated each iteration
    # The variable must also be included in the outputs
    # TODO ask John about this issue, how common is it to have variables like this

    import numpy as np
    import math

    # Assign Parameter Values #
    a = self.parameters['a']
    b = self.parameters['b']
    b0 = self.parameters['b0']
    c = self.parameters['c']
    L = self.parameters['L']
    KDMIMILK = self.parameters['KDMIMILK']
    KDMImbw = self.parameters['KDMImbw']
    Hcfeed = self.parameters['Hcfeed']
    Hcmilk = self.parameters['Hcmilk']
    KL = self.parameters['KL']
    expL = self.parameters['expL']
    pregdate = self.parameters['pregdate']
    kf1 = self.parameters['kf1']
    kf2 = self.parameters['kf2']
    milk_value = self.parameters['milk_value']

    # Paramaters that require an initial value #
    if t == 0:
            E = self.parameters['E']
    else:
        E = prev_var_return[5]


    # Variables w/ Differential Equation #
    BW = stateVars[0]
    ER = stateVars[1]
    milkcumul = stateVars[2]
    DMIcumul = stateVars[3]
    IOFCcumul = stateVars[4]

    # Model Equations # 
    feed_price = (101 * Hcfeed + 2.7) / 1000

    BFat = ER/9.0       
    # Body fat

    Lmod = 1.0-(1.0-L)/(1.0+(KL/BFat)**expL)         
        # 1 - (1-L) = L?
        # Modifying value of L? Why the adjustment?

    dijkstra_milk = a * np.exp(b * (1 - np.exp(-b0 * t)) / b0 - c * t)
    # John has modified S to give 4% FCM yield per alveolus 

    NEmilk = E**Lmod*dijkstra_milk                         
        # Net energy milk
        # On slide 28 this is equation for FCM yield but L is modified here
        # Is L modified to give a NEmilk as well as a milk yield (milk)

    milk = NEmilk/Hcmilk                
        # Milk production

    DMINRC = (KDMIMILK * milk + KDMImbw * BW**0.75)*(1.0-math.exp(-0.192*(t/7+3.67)))   
        # Slide 28
        # NRC DMI equation

    fdbk = (kf1*t/7+kf2)*(ER-self.iStateVars[1])/self.iStateVars[1]       
    # Ellis 2006
        # Feedback on DMI

    DMI = DMINRC - fdbk                             
        # Dry matter intake     

    NEI = Hcfeed*DMI                                
        # Net energy intake

    NEmaint = 0.096*BW**0.75                        
        # Net energy maintenance

    if t < pregdate + 190:                          
        NEgest = 0
    else:
        NEgest = (0.00318*(t-pregdate-190)-0.0352)/0.218
        # Net energy gestation

    NEreqt = NEmaint + dijkstra_milk + NEgest + 5.12*2.0          
        # NE requirement, from NRC

    E = NEI/NEreqt            
        # Energy balance

    # Differential Equations # 
    dERdt = NEI - NEmaint - NEmilk - NEgest
    # Energy requirement (NE)

    if dERdt < 0:
        dBWdt = dERdt/4.92  # 4.92 Mcal/kg in NRC(1988)
    else:
        dBWdt = dERdt/5.12  # 5.12 Mcal/kg for gain
        # Bodyweight gain

    IOFC = (milk * milk_value) - (DMI * feed_price)

    # Format data for return # 
    differential_return = [dBWdt, dERdt, milk, DMI, IOFC]
    local_variables = locals()
    # Store local variables 
    variable_returns = [local_variables.get(variable_name) for variable_name in self.outputs]
    # Create list of variables to return

    return differential_return, variable_returns

functions_to_include = [laccurve5_dijkstra]


In [3]:
# Setup a herd
herd = moo.Herd(outputs=output)

# Add 1 cow to the herd
herd.add_cow(moo.Cow(name='test_cow', iStateVars=iStateVars, parameters=parameters_laccurve5, outputs=herd.outputs, model_functions=functions_to_include))

# herd.run_all_models()


In [23]:
# Create individual cow, cow_2
cow_2 = moo.Cow(name='Daisy', iStateVars=iStateVars, parameters=parameters_laccurve5, outputs=output, model_functions=functions_to_include)

# run model on cow_2
cow_2.execute_runModel(0, 400, 0.001, 7, 0)


Running Model....
This is the scenario_param: None
          t  dijkstra_milk     NEmilk       milk        DMI         E  \
0     0.000      18.308587  12.358644  16.500192   9.405910  0.367896   
1     6.999      24.120227  14.222020  18.988011  11.460375  0.393894   
2    13.999      28.988849  18.147621  24.229133  14.024893  0.437774   
3    20.999      32.704232  21.508099  28.715753  16.361903  0.477539   
4    27.999      35.322751  24.201806  32.312158  18.382754  0.513327   
5    34.999      37.023175  26.246354  35.041861  20.057686  0.545129   
6    41.999      38.011797  27.721101  37.010815  21.398909  0.573043   
7    48.999      38.475360  28.726387  38.352987  22.441821  0.597293   
8    55.999      38.564762  29.360339  39.199384  23.230788  0.618201   
9    62.999      38.394358  29.708383  39.664062  23.810435  0.636133   
10   69.999      38.047444  29.840256  39.840128  24.221225  0.651467   
11   76.999      37.583167  29.810737  39.800717  24.497786  0.664567   


In [25]:
cow_3 = moo.Cow(name='Bella', iStateVars=iStateVars, parameters=parameters_laccurve5, outputs=output, model_functions=functions_to_include)

# run model on cow_2
cow_3.execute_runModel(0, 400, 0.001, 7, 0)

Running Model....
This is the scenario_param: None
          t  dijkstra_milk     NEmilk       milk        DMI         E  \
0     0.000      18.308587  12.358644  16.500192   9.405910  0.367896   
1     6.999      24.120227  14.222020  18.988011  11.460375  0.393894   
2    13.999      28.988849  18.147621  24.229133  14.024893  0.437774   
3    20.999      32.704232  21.508099  28.715753  16.361903  0.477539   
4    27.999      35.322751  24.201806  32.312158  18.382754  0.513327   
5    34.999      37.023175  26.246354  35.041861  20.057686  0.545129   
6    41.999      38.011797  27.721101  37.010815  21.398909  0.573043   
7    48.999      38.475360  28.726387  38.352987  22.441821  0.597293   
8    55.999      38.564762  29.360339  39.199384  23.230788  0.618201   
9    62.999      38.394358  29.708383  39.664062  23.810435  0.636133   
10   69.999      38.047444  29.840256  39.840128  24.221225  0.651467   
11   76.999      37.583167  29.810737  39.800717  24.497786  0.664567   


In [20]:
df1 = cow_2.results[0]
df2 = cow_3.results[0]